In [1]:
from strategies import *
from scipy.signal import savgol_filter
import pandas as pd

if __name__ == '__main__':
    
    df = pd.read_csv("../../backtesting/data/GOOGL/GOOGL.USUSD_Candlestick_5_M_ASK_05.10.2022-05.10.2024.csv")

    df['Gmt time']=df["Gmt time"].str.replace(".000","")
    df['Gmt time']=pd.to_datetime(df['Gmt time'],format='%d.%m.%Y %H:%M:%S')
    df['YMD'] = df['Gmt time'].dt.strftime('%Y%m%d')
    df.set_index("YMD")
    
    df.rename(columns={"Open": "open"}, inplace=True)
    df.rename(columns={"High": "high"}, inplace=True)
    df.rename(columns={"Low": "low"}, inplace=True)
    df.rename(columns={"Volume": "volume"}, inplace=True)
    df.rename(columns={"Close": "close"}, inplace=True)
    df.rename(columns={'Gmt time':'datetime_gmt'}, inplace = True)
    
    df['datetime_est']=pd.to_datetime(df["datetime_gmt"], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern')
    df["close_smooth"] = savgol_filter(df.close, 49, 5)
    df["high_smooth"] = savgol_filter(df.high, 49, 5)
    df["low_smooth"] = savgol_filter(df.low, 49, 5)
    
    fromTodayStart = '2022-10-05 09:30:00'
    toNow   = '2022-12-05 16:00:00'
    df = df[df['datetime_est'].between(fromTodayStart, toNow)].copy()

    df = df[df.notnull().all(axis=1)]
    df=df[(df.volume != 0)]
    df=df[df.high!=df.low]

    df.reset_index(drop=True, inplace=True)
    
    df = df[['datetime_est','YMD','high','low','close','close_smooth','high_smooth','low_smooth']] 

    result = squeez(df)
    print(result)

      position
0            0
1            0
2            0
3            0
4            0
...        ...
3289         2
3290         2
3291         2
3292         2
3293         0

[3294 rows x 1 columns]
